# 🧬 Helix — Real Analysis on YOUR data (Colab)

Upload any tabular **CSV** (e.g. from Kaggle), pick the **target column**, and Helix runs a
*genuine* end-to-end analysis — not a canned demo:

**clean → encode → FLAML AutoML → real metrics → SHAP explainability → LLM report**,
narrated live by the 7 agents, with **real self-correction** on messy columns.

### Recommended test dataset
**Telco Customer Churn** — https://www.kaggle.com/datasets/blastchar/telco-customer-churn
(download the single CSV; target column = `Churn`). It has a genuinely messy `TotalCharges`
column that triggers real auto-correction.

### Use a FREE LLM (optional but recommended)
Paste a free **Groq** key in the config cell to get real LLM plans/reports. Without a key the
agents use an offline mock — **the ML analysis still runs for real either way**.

### ▶ Steps
1. **Runtime → Run all**  (first cell installs FLAML/SHAP — ~1 min)
2. Scroll to the Gradio app, **upload your CSV**, pick the **target**, click **Run real analysis**


In [ ]:
!pip install -q gradio "flaml[automl]" shap scikit-learn requests RestrictedPython chromadb
print("✅ dependencies installed")

In [ ]:
import os

# ── FREE-tier API model (recommended: Groq). Paste your key to enable real LLM. ──
# Get a key (free, no card): https://console.groq.com/keys
os.environ["HELIX_LLM_BASE_URL"] = "https://api.groq.com/openai/v1"
os.environ["HELIX_LLM_API_KEY"]  = ""   # <-- paste gsk_... here (leave blank = offline mock)
os.environ["HELIX_LLM_MODEL"]    = "llama-3.3-70b-versatile"

# Alternatives (free tier, OpenAI-compatible):
#   Gemini:     base "https://generativelanguage.googleapis.com/v1beta/openai", model "gemini-2.0-flash"
#   OpenRouter: base "https://openrouter.ai/api/v1", model "deepseek/deepseek-chat-v3-0324:free"

print("LLM:", ("REAL - " + os.getenv("HELIX_LLM_MODEL")) if os.getenv("HELIX_LLM_API_KEY") else "MOCK (offline; paste a key above for real LLM)")

In [ ]:
import os, requests

class MockLLM:
    is_mock = True
    def complete(self, role, context):
        if role == "planner":  return "\n".join(context.get("plan", []))
        if role == "coder":    return "\n".join(context.get("code", []))
        if role == "reporter":
            return "\n\n".join(context.get("report", [])) + "\n\nRecommendation: " + context.get("recommendation", "")
        return ""

SYSTEM = {
    "planner": "You are a senior data scientist. Output a concise numbered analysis plan (max 6 steps), one per line, no prose.",
    "coder": "You are an expert Python data scientist. Output only runnable Python, no prose.",
    "reporter": "You are a business analyst. Write a short plain-English narrative of the findings, then a one-line recommendation.",
}

def _prompt(role, ctx):
    ds = ctx.get("dataset", {})
    head = "Dataset: " + str(ds.get("name","?")) + " (" + str(ds.get("rows","?")) + " rows x " + str(ds.get("cols","?")) + " cols). Task: " + str(ds.get("task","?")) + ". Target: " + str(ds.get("target","?")) + ". "
    if role == "planner":  return head + "Goal: " + ctx.get("goal","") + ". Write the plan."
    if role == "coder":    return head + "Write Python to load and model the data."
    if role == "reporter": return head + "Metrics: " + ctx.get("metrics","") + ". Drivers: " + ctx.get("drivers","") + ". Write the report + a one-line recommendation."
    return head

class APILLM:
    is_mock = False
    def __init__(self, base, key, model):
        self.base = base.rstrip("/"); self.key = key; self.model = model
    def complete(self, role, ctx):
        h = {"Content-Type": "application/json", "Authorization": "Bearer " + self.key}
        body = {"model": self.model, "temperature": 0.2,
                "messages": [{"role":"system","content":SYSTEM.get(role,"")},{"role":"user","content":_prompt(role,ctx)}]}
        r = requests.post(self.base + "/chat/completions", json=body, headers=h, timeout=60)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"].strip()

class AnthropicLLM:
    is_mock = False
    def __init__(self, base, key, model):
        self.base = base.rstrip("/"); self.key = key; self.model = model
    def complete(self, role, ctx):
        h = {"x-api-key": self.key, "anthropic-version": "2023-06-01", "content-type": "application/json"}
        body = {"model": self.model, "max_tokens": 1024, "system": SYSTEM.get(role, ""), "messages": [{"role": "user", "content": _prompt(role, ctx)}]}
        r = requests.post(self.base + "/messages", json=body, headers=h, timeout=60)
        r.raise_for_status()
        return r.json()["content"][0]["text"].strip()

PROVIDERS = {
    "groq": ("https://api.groq.com/openai/v1", "openai", "llama-3.3-70b-versatile"),
    "openai": ("https://api.openai.com/v1", "openai", "gpt-4o-mini"),
    "deepseek": ("https://api.deepseek.com", "openai", "deepseek-chat"),
    "mistral": ("https://api.mistral.ai/v1", "openai", "mistral-small-latest"),
    "openrouter": ("https://openrouter.ai/api/v1", "openai", "deepseek/deepseek-chat"),
    "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai", "openai", "gemini-2.0-flash"),
    "anthropic": ("https://api.anthropic.com/v1", "anthropic", "claude-3-5-haiku-latest"),
}

def make_llm(provider, model, key):
    if not key:
        return MockLLM()
    base, kind, default = PROVIDERS.get(provider, PROVIDERS["groq"])
    model = model or default
    return AnthropicLLM(base, key, model) if kind == "anthropic" else APILLM(base, key, model)

def get_llm(role):
    base = os.getenv("HELIX_" + role.upper() + "_BASE_URL") or os.getenv("HELIX_LLM_BASE_URL", "")
    key  = os.getenv("HELIX_" + role.upper() + "_API_KEY")  or os.getenv("HELIX_LLM_API_KEY", "")
    model = os.getenv("HELIX_" + role.upper() + "_MODEL")   or os.getenv("HELIX_LLM_MODEL", "")
    if key:
        return APILLM(base or "https://api.groq.com/openai/v1", key, model or "llama-3.3-70b-versatile")
    return MockLLM()

print("LLM layer ready -", "MOCK" if get_llm("planner").is_mock else "REAL")

In [ ]:
# === Real analysis engine (identical to the local backend) ===
"""Real tabular analysis engine.

Takes an arbitrary CSV (as a DataFrame) + a target column and runs a genuine
ML workflow: clean → encode → train (FLAML AutoML, sklearn fallback) → evaluate
→ explain (SHAP, importance fallback) → EDA. Returns results in the same shape
the frontend already renders (``RunResults``), plus the list of cleaning
"fixes" applied (which the agents narrate as self-correction).

Everything is wrapped defensively so it degrades gracefully on messy data
instead of crashing.
"""

from __future__ import annotations

import math
from typing import Any

import numpy as np
import pandas as pd


def _is_classification(y: pd.Series) -> bool:
    if y.dtype == object or str(y.dtype) == "category":
        return True
    if pd.api.types.is_float_dtype(y) and y.nunique() > 15:
        return False
    return y.nunique() <= 15


def clean(df: pd.DataFrame, target: str) -> tuple[pd.DataFrame, list[str]]:
    fixes: list[str] = []
    df = df.copy()

    stripped = {c: c.strip() for c in df.columns}
    if any(k != v for k, v in stripped.items()):
        df = df.rename(columns=stripped)
        fixes.append("stripped whitespace from column names")
    target = target.strip()

    # coerce numeric-looking object columns (e.g. Telco 'TotalCharges' has blanks)
    for col in df.columns:
        if col == target or df[col].dtype != object:
            continue
        coerced = pd.to_numeric(df[col], errors="coerce")
        valid = coerced.notna().sum()
        if valid >= 0.8 * len(df) and valid > 0:
            bad = int(df[col].notna().sum() - valid)
            df[col] = coerced
            if bad:
                fixes.append(f"coerced '{col}' to numeric ({bad} bad values -> NaN)")

    # drop id-like columns (name-based, or all-unique *string* keys — not continuous numbers)
    id_cols = [
        c
        for c in df.columns
        if c != target
        and (
            c.lower() == "id"
            or c.lower().endswith("id")
            or (
                df[c].dtype == object
                and df[c].nunique(dropna=False) == len(df)
                and df[c].astype(str).str.len().mean() < 25  # short keys, not free text
            )
        )
    ]
    if id_cols:
        df = df.drop(columns=id_cols)
        fixes.append("dropped identifier column(s): " + ", ".join(id_cols))

    # drop rows with a missing target
    before = len(df)
    df = df.dropna(subset=[target])
    if len(df) < before:
        fixes.append(f"dropped {before - len(df)} rows with a missing target")

    return df, fixes


def analyze_dataframe(
    df: pd.DataFrame,
    target: str,
    task: str = "auto",
    time_budget: int = 20,
) -> dict[str, Any]:
    df.columns = [c.strip() for c in df.columns]
    if task == "clustering":
        return _cluster(df)
    target = target.strip()
    if target not in df.columns:
        raise ValueError(f"target column '{target}' not found in dataset")

    df, fixes = clean(df, target)

    # sample very large data for speed
    if len(df) > 30000:
        df = df.sample(30000, random_state=42)
        fixes.append("sampled 30,000 rows for a fast run")

    y_raw = df[target]
    is_clf = _is_classification(y_raw) if task in ("auto", "nlp") else (task == "classification")
    X = df.drop(columns=[target]).copy()

    # NLP: detect & TF-IDF a free-text feature column
    text_terms: list[tuple[str, float]] = []
    for c in list(X.columns):
        if X[c].dtype == object:
            s = X[c].dropna().astype(str)
            if len(s) and s.str.len().mean() > 20 and s.nunique() >= 15:
                X, text_terms = _vectorize_text(X, c)
                fixes.append(f"vectorized text column '{c}' with TF-IDF")
                break

    # parse date columns into numeric parts (universal across domains)
    X = _extract_dates(X, fixes)

    # impute + encode features
    imputed = False
    for c in X.columns:
        if X[c].dtype == object or str(X[c].dtype) == "category":
            X[c] = X[c].astype("category").cat.codes  # NaN → -1
        else:
            if X[c].isna().any():
                X[c] = X[c].fillna(X[c].median())
                imputed = True
    if imputed:
        fixes.append("imputed missing numeric values with the median")
    n_cat = int(sum(df[c].dtype == object for c in df.columns if c != target))
    if n_cat:
        fixes.append(f"label-encoded {n_cat} categorical feature(s)")

    features = list(X.columns)

    # target encoding
    classes: list[Any] = []
    positive_label = None
    if is_clf:
        ycat = y_raw.astype("category")
        classes = list(ycat.cat.categories)
        y = ycat.cat.codes.to_numpy()
        binary = len(classes) == 2
        if binary:
            positive_label = classes[1]
    else:
        y = pd.to_numeric(y_raw, errors="coerce").to_numpy()
        binary = False

    from sklearn.model_selection import train_test_split

    strat = y if (is_clf and pd.Series(y).value_counts().min() >= 2) else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=strat
    )

    best_model = "model"
    proba = None
    try:
        from flaml import AutoML

        automl = AutoML()
        metric = ("roc_auc" if binary else "accuracy") if is_clf else "r2"
        automl.fit(
            X_train,
            y_train,
            task="classification" if is_clf else "regression",
            time_budget=time_budget,
            metric=metric,
            verbose=0,
            early_stop=True,
        )
        model = automl.model.estimator
        best_model = {
            "lgbm": "LightGBM",
            "xgboost": "XGBoost",
            "rf": "Random Forest",
            "extra_tree": "Extra Trees",
            "catboost": "CatBoost",
            "lrl1": "Logistic Regression",
            "lrl2": "Logistic Regression",
        }.get(str(automl.best_estimator), str(automl.best_estimator))
        preds = automl.predict(X_test)
        if is_clf:
            try:
                proba = automl.predict_proba(X_test)
            except Exception:
                proba = None
    except Exception:
        from sklearn.ensemble import (
            HistGradientBoostingClassifier,
            HistGradientBoostingRegressor,
        )

        if is_clf:
            model = HistGradientBoostingClassifier(random_state=42).fit(X_train, y_train)
            preds = model.predict(X_test)
            try:
                proba = model.predict_proba(X_test)
            except Exception:
                proba = None
        else:
            model = HistGradientBoostingRegressor(random_state=42).fit(X_train, y_train)
            preds = model.predict(X_test)
        best_model = "HistGradientBoosting"

    metrics, headline, task_label = _score(is_clf, binary, y_test, preds, proba)
    bars, bars_title = _importance(model, X, X_test, y_test, y, features, is_clf)
    if text_terms:  # NLP: surface themes instead of opaque TF-IDF dims
        bars = [{"label": t, "value": v} for t, v in text_terms]
        bars_title = "Top themes"
        task_label = "NLP + analytics"
    dist, dist_title = _eda(df, target, features, bars, is_clf, positive_label, y_raw)

    report, recommendation = _default_report(
        target, task_label, headline, bars, dist, dist_title
    )

    return {
        "taskLabel": task_label,
        "bestModel": best_model,
        "headline": headline,
        "metrics": metrics,
        "barsTitle": bars_title,
        "bars": bars,
        "distTitle": dist_title,
        "dist": dist,
        "report": report,
        "recommendation": recommendation,
        # extra metadata for the agents / self-heal narration
        "_fixes": fixes,
        "_features": features,
        "_target": target,
        "_rows": int(len(df)),
        "_cols": int(df.shape[1]),
        "_drivers": [
            {"feature": b["label"], "importance": round(b["value"], 2), "direction": b.get("sign", 0)}
            for b in bars[:4]
        ],
    }


def _score(is_clf, binary, y_test, preds, proba):
    if is_clf:
        from sklearn.metrics import (
            accuracy_score,
            f1_score,
            precision_score,
            recall_score,
            roc_auc_score,
        )

        acc = accuracy_score(y_test, preds)
        metrics = [{"label": "Accuracy", "value": f"{acc:.2f}"}]
        head_frac, head_val, head_lbl = acc, f"{acc:.2f}", "Accuracy"
        if binary and proba is not None:
            try:
                auc = roc_auc_score(y_test, proba[:, 1])
                metrics.append({"label": "ROC-AUC", "value": f"{auc:.2f}"})
                head_frac, head_val, head_lbl = auc, f"{auc:.2f}", "ROC-AUC"
            except Exception:
                pass
        avg = "binary" if binary else "macro"
        metrics.append({"label": "Precision", "value": f"{precision_score(y_test, preds, average=avg, zero_division=0):.2f}"})
        metrics.append({"label": "Recall", "value": f"{recall_score(y_test, preds, average=avg, zero_division=0):.2f}"})
        metrics.append({"label": "F1", "value": f"{f1_score(y_test, preds, average=avg, zero_division=0):.2f}"})
        return metrics[:4], {"fraction": float(max(0, min(1, head_frac))), "value": head_val, "label": head_lbl}, "Binary classification" if binary else "Multiclass classification"

    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    r2 = r2_score(y_test, preds)
    rmse = math.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    with np.errstate(divide="ignore", invalid="ignore"):
        denom = np.where(np.abs(y_test) < 1e-9, np.nan, y_test)
        mape = np.nanmean(np.abs((y_test - preds) / denom)) * 100
    metrics = [
        {"label": "R2", "value": f"{r2:.2f}"},
        {"label": "RMSE", "value": f"{rmse:.0f}" if rmse >= 10 else f"{rmse:.2f}"},
        {"label": "MAE", "value": f"{mae:.0f}" if mae >= 10 else f"{mae:.2f}"},
        {"label": "MAPE", "value": f"{mape:.1f}%" if not math.isnan(mape) else "—"},
    ]
    return metrics, {"fraction": float(max(0, min(1, r2))), "value": f"{r2:.2f}", "label": "R2 score"}, "Regression"


def _importance(model, X, X_test, y_test, y, features, is_clf):
    importances = None
    signs = None
    title = "Feature importance"
    try:
        import shap

        sample = X_test.sample(min(200, len(X_test)), random_state=42)
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(sample)
        arr = sv[1] if isinstance(sv, list) and len(sv) > 1 else (sv[0] if isinstance(sv, list) else sv)
        arr = np.asarray(arr)
        if arr.ndim == 3:  # (n, features, classes)
            arr = arr[:, :, -1]
        importances = np.abs(arr).mean(axis=0)
        signs = np.sign(arr.mean(axis=0))
        title = "SHAP feature importance"
    except Exception:
        try:
            importances = np.asarray(model.feature_importances_, dtype=float)
        except Exception:
            from sklearn.inspection import permutation_importance

            pi = permutation_importance(model, X_test, y_test, n_repeats=4, random_state=42)
            importances = pi.importances_mean
        signs = []
        for c in features:
            try:
                corr = np.corrcoef(X[c].to_numpy(dtype=float), np.asarray(y, dtype=float))[0, 1]
                signs.append(np.sign(corr) if not math.isnan(corr) else 1)
            except Exception:
                signs.append(1)
        signs = np.asarray(signs)

    importances = np.nan_to_num(np.asarray(importances, dtype=float))
    order = np.argsort(importances)[::-1][:6]
    top = importances[order[0]] if len(order) and importances[order[0]] > 0 else 1.0
    bars = [
        {
            "label": str(features[i])[:22],
            "value": float(max(0.04, min(1.0, importances[i] / top))),
            "sign": int(signs[i]) if signs is not None and signs[i] != 0 else 1,
        }
        for i in order
    ]
    return bars, title


def _eda(df, target, features, bars, is_clf, positive_label, y_raw):
    try:
        if is_clf and positive_label is not None:
            cand = [
                b["label"]
                for b in bars
                if b["label"] in df.columns and df[b["label"]].nunique() <= 8
            ]
            col = cand[0] if cand else None
            if col is None:
                for c in df.columns:
                    if c != target and df[c].nunique() <= 8:
                        col = c
                        break
            if col is not None:
                rate = df.groupby(col)[target].apply(lambda s: (s == positive_label).mean())
                rate = rate.sort_values(ascending=False).head(5)
                items = [
                    {"label": str(k), "value": float(v), "display": f"{v * 100:.0f}%"}
                    for k, v in rate.items()
                ]
                return items, f"{target} rate by {col}"
        else:
            num = [b["label"] for b in bars if b["label"] in df.columns and pd.api.types.is_numeric_dtype(df[b["label"]])]
            col = num[0] if num else None
            if col is not None:
                binned = pd.qcut(df[col], q=4, duplicates="drop")
                grp = df.groupby(binned, observed=True)[target].mean()
                m = grp.max() or 1
                items = [
                    {"label": str(k), "value": float(v / m), "display": f"{v:.0f}" if v >= 10 else f"{v:.2f}"}
                    for k, v in grp.items()
                ]
                return items, f"Average {target} by {col}"
    except Exception:
        pass
    # fallback: target distribution
    vc = y_raw.value_counts(normalize=True).head(5)
    items = [{"label": str(k), "value": float(v), "display": f"{v * 100:.0f}%"} for k, v in vc.items()]
    return items, f"{target} distribution"


def _default_report(target, task_label, headline, bars, dist, dist_title):
    top = bars[0]["label"] if bars else "the top feature"
    second = bars[1]["label"] if len(bars) > 1 else "other factors"
    report = [
        f"The {task_label.lower()} model reached {headline['value']} {headline['label']} on held-out data. {top} is the single strongest driver of {target}, followed by {second}.",
        f"The breakdown '{dist_title}' shows where {target} concentrates, giving a clear, data-backed view to act on.",
    ]
    recommendation = f"Prioritise {top} — it has the largest measured effect on {target}."
    return report, recommendation


def _extract_dates(X: pd.DataFrame, fixes: list[str]) -> pd.DataFrame:
    """Turn date-like text columns into numeric year/month/day/weekday features."""
    import warnings as _w

    for c in list(X.columns):
        if X[c].dtype != object:
            continue
        s = X[c].dropna().astype(str)
        if not len(s) or s.str.len().mean() > 30:
            continue
        if s.str.contains(r"[-/:]", regex=True).mean() < 0.6:
            continue
        with _w.catch_warnings():
            _w.simplefilter("ignore")
            parsed = pd.to_datetime(X[c], errors="coerce")
        if parsed.notna().mean() > 0.8:
            X[f"{c}_year"] = parsed.dt.year
            X[f"{c}_month"] = parsed.dt.month
            X[f"{c}_day"] = parsed.dt.day
            X[f"{c}_dow"] = parsed.dt.dayofweek
            X = X.drop(columns=[c])
            fixes.append(f"parsed '{c}' as datetime -> year/month/day/weekday")
    return X


def _vectorize_text(X: pd.DataFrame, col: str):
    """TF-IDF a free-text column; return (new_X, [(term, weight), ...])."""
    from sklearn.feature_extraction.text import TfidfVectorizer

    vec = TfidfVectorizer(max_features=60, stop_words="english")
    mat = vec.fit_transform(X[col].fillna("").astype(str))
    means = np.asarray(mat.mean(axis=0)).ravel()
    vocab = vec.get_feature_names_out()
    order = means.argsort()[::-1][:6]
    top = float(means[order[0]]) if len(order) and means[order[0]] > 0 else 1.0
    terms = [(str(vocab[i]), float(max(0.08, means[i] / top))) for i in order]
    tf = pd.DataFrame(
        mat.toarray(),
        columns=[f"tf_{i}" for i in range(mat.shape[1])],
        index=X.index,
    )
    return X.drop(columns=[col]).join(tf), terms


def _cluster(df: pd.DataFrame) -> dict[str, Any]:
    """Unsupervised KMeans with silhouette-based k selection."""
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    from sklearn.preprocessing import StandardScaler

    fixes: list[str] = []
    work = df.copy()
    ids = [
        c
        for c in work.columns
        if c.lower() == "id"
        or c.lower().endswith("id")
        or (
            work[c].dtype == object
            and work[c].nunique(dropna=False) == len(work)
            and work[c].astype(str).str.len().mean() < 25
        )
    ]
    if ids:
        work = work.drop(columns=ids)
        fixes.append("dropped identifier column(s): " + ", ".join(ids))
    for c in work.columns:
        if work[c].dtype == object:
            num = pd.to_numeric(work[c], errors="coerce")
            work[c] = num if num.notna().sum() >= 0.8 * len(work) else work[c].astype("category").cat.codes
    work = work.fillna(work.median(numeric_only=True)).select_dtypes(include="number")
    if work.shape[1] < 2:
        raise ValueError("clustering needs at least 2 numeric features")
    if len(work) > 20000:
        work = work.sample(20000, random_state=42)
        fixes.append("sampled 20,000 rows for a fast run")

    Xs = StandardScaler().fit_transform(work)
    best_k, best_sil, best_labels = 2, -1.0, None
    for k in range(2, min(9, len(work))):
        labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(Xs)
        sil = silhouette_score(Xs, labels)
        if sil > best_sil:
            best_k, best_sil, best_labels = k, sil, labels
    fixes.append(f"label-encoded {sum(df[c].dtype == object for c in df.columns)} categorical feature(s)")

    sizes = pd.Series(best_labels).value_counts(normalize=True).sort_index()
    dist = [
        {"label": f"Segment {i + 1}", "value": float(v), "display": f"{v * 100:.0f}%"}
        for i, v in enumerate(sizes)
    ]
    grouped = work.reset_index(drop=True).assign(_c=best_labels)
    sep = {c: float(grouped.groupby("_c")[c].mean().var()) for c in work.columns}
    order = sorted(sep, key=lambda c: -sep[c])[:6]
    mx = max(sep.values()) or 1.0
    bars = [{"label": str(c)[:22], "value": float(max(0.08, sep[c] / mx))} for c in order]
    second = order[1] if len(order) > 1 else order[0]

    return {
        "taskLabel": "Clustering",
        "bestModel": f"KMeans (k={best_k})",
        "headline": {"value": str(best_k), "label": "segments"},
        "metrics": [
            {"label": "Clusters", "value": str(best_k)},
            {"label": "Silhouette", "value": f"{best_sil:.2f}"},
            {"label": "Features", "value": str(work.shape[1])},
            {"label": "Samples", "value": str(len(work))},
        ],
        "barsTitle": "Defining dimensions",
        "bars": bars,
        "distTitle": "Segment sizes",
        "dist": dist,
        "report": [
            f"K-Means found {best_k} natural segments (silhouette {best_sil:.2f}). They separate most along {order[0]} and {second}.",
            "Each segment groups rows with a similar profile — ready for targeted strategies.",
        ],
        "recommendation": f"Profile the largest segment and differentiate strategy along {order[0]}.",
        "_fixes": fixes,
        "_features": list(work.columns),
        "_target": "(unsupervised)",
        "_rows": int(len(work)),
        "_cols": int(df.shape[1]),
        "_drivers": [{"feature": b["label"], "importance": round(b["value"], 2), "direction": 0} for b in bars[:4]],
    }


In [ ]:
# === RestrictedPython sandbox (Phase 3) — identical to the backend ===
"""RestrictedPython execution sandbox (Phase 3).

Runs LLM-generated Python with file-system, network, and dangerous imports
blocked. Captures stdout and tracebacks so the Critic can read failures and
self-correct — the literal "self-correcting code execution" from the proposal.

Safety model: RestrictedPython compiles the code with guarded attribute/item/
iteration access, a curated builtins set, and an import hook that only allows a
data-science whitelist (no os/sys/socket/subprocess/open).
"""

from __future__ import annotations

import builtins as _py_builtins
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
from RestrictedPython import compile_restricted, safe_builtins, utility_builtins
from RestrictedPython.Eval import default_guarded_getitem, default_guarded_getiter
from RestrictedPython.Guards import (
    full_write_guard,
    guarded_iter_unpack_sequence,
    guarded_unpack_sequence,
    safer_getattr,
)
from RestrictedPython.PrintCollector import PrintCollector

# Modules the generated code is allowed to import.
_ALLOWED_IMPORTS = {
    "pandas",
    "numpy",
    "math",
    "statistics",
    "random",
    "datetime",
    "collections",
    "itertools",
    "functools",
    "re",
    "json",
    "sklearn",
    "scipy",
}

# Extra safe builtins data-science code commonly needs (beyond safe_builtins).
_EXTRA_BUILTINS = (
    "enumerate", "sorted", "sum", "min", "max", "list", "dict", "set",
    "tuple", "map", "filter", "reversed", "any", "all", "len", "range",
    "abs", "round", "str", "int", "float", "bool", "zip", "isinstance",
)


def _safe_import(name, *args, **kwargs):
    root = name.split(".")[0]
    if root in _ALLOWED_IMPORTS:
        return __import__(name, *args, **kwargs)
    raise ImportError(f"import of '{name}' is blocked in the sandbox")


@dataclass
class SandboxResult:
    ok: bool
    stdout: str
    error: str  # short, last line of the traceback when ok is False


def run_in_sandbox(code: str, variables: dict | None = None) -> SandboxResult:
    """Compile + execute ``code`` under RestrictedPython. Never raises."""
    safe = dict(safe_builtins)
    safe.update(utility_builtins)
    for name in _EXTRA_BUILTINS:
        safe.setdefault(name, getattr(_py_builtins, name))
    safe["__import__"] = _safe_import

    glb = {
        "__builtins__": safe,
        "_print_": PrintCollector,
        "_getattr_": safer_getattr,
        "_getitem_": default_guarded_getitem,
        "_getiter_": default_guarded_getiter,
        "_iter_unpack_sequence_": guarded_iter_unpack_sequence,
        "_unpack_sequence_": guarded_unpack_sequence,
        "_write_": full_write_guard,
        "pd": pd,
        "np": np,
    }
    if variables:
        glb.update(variables)

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            byte_code = compile_restricted(code, "<agent-code>", "exec")
    except SyntaxError as exc:
        return SandboxResult(False, "", f"SyntaxError: {exc}")

    try:
        exec(byte_code, glb)  # noqa: S102 — sandboxed
        out = glb["_print"]() if "_print" in glb else ""
        return SandboxResult(True, out, "")
    except Exception as exc:  # noqa: BLE001 — sandboxed; report, never raise
        out = glb["_print"]() if "_print" in glb else ""
        msg = str(exc)
        if len(msg) > 90:  # pandas dumps whole columns into some errors
            msg = msg[:60].rstrip() + " ... " + msg[-25:].lstrip()
        return SandboxResult(False, out, f"{type(exc).__name__}: {msg}")


def strip_code_fences(text: str) -> str:
    """Strip ```python ... ``` fences that LLMs often wrap code in."""
    t = text.strip()
    if t.startswith("```"):
        lines = t.splitlines()
        lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        t = "\n".join(lines)
    return t.strip()


In [ ]:
# === ChromaDB RAG (Phase 5) — identical to the backend ===
"""Lightweight RAG over data-science documentation (Phase 5).

A curated corpus of pandas / scikit-learn / FLAML / SHAP recipes is embedded
with a local MiniLM model (ChromaDB's default, ONNX — no GPU) and stored in
ChromaDB. The Coder retrieves the most relevant snippets to ground its code,
cutting hallucinated APIs.

Falls back to a keyword match if ChromaDB/embeddings are unavailable, so the
pipeline never breaks.
"""

from __future__ import annotations

_DOCS: list[tuple[str, str]] = [
    ("pandas-missing", "Handle missing values: fill numeric columns with df.fillna(df.median(numeric_only=True)); fill categoricals with the mode df['c'].fillna(df['c'].mode()[0])."),
    ("pandas-encode", "Encode categorical features for tree models with label codes: df['c'] = df['c'].astype('category').cat.codes; or one-hot via pd.get_dummies(df, columns=cols)."),
    ("pandas-numeric", "Coerce a numeric column stored as text: pd.to_numeric(df['c'], errors='coerce'). Aggregate only numeric columns with df.mean(numeric_only=True) to avoid TypeErrors."),
    ("flaml-automl", "FLAML AutoML: from flaml import AutoML; m=AutoML(); m.fit(X_train, y_train, task='classification', time_budget=30, metric='roc_auc'); preds=m.predict(X_test)."),
    ("sklearn-split", "Train/test split: from sklearn.model_selection import train_test_split; X_tr,X_te,y_tr,y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)."),
    ("sklearn-metrics", "Classification metrics: accuracy_score, roc_auc_score(y, proba[:,1]), precision_score, recall_score, f1_score. Regression: r2_score, mean_squared_error, mean_absolute_error."),
    ("shap-explain", "Explain a tree model with SHAP: import shap; e=shap.TreeExplainer(model); vals=e.shap_values(X_sample); rank features by abs(vals).mean(0); sign by vals.mean(0)."),
    ("eda-basic", "Quick EDA: df.shape, df.describe(), df['target'].value_counts(normalize=True), and df.corr(numeric_only=True) to find correlations with the target."),
]

_collection = None
_failed = False


def _get_collection():
    global _collection, _failed
    if _collection is not None or _failed:
        return _collection
    try:
        import chromadb

        client = chromadb.Client()
        col = client.get_or_create_collection("ds_docs")
        if col.count() == 0:
            col.add(ids=[d[0] for d in _DOCS], documents=[d[1] for d in _DOCS])
        _collection = col
    except Exception:
        _failed = True
        _collection = None
    return _collection


def _keyword_fallback(query: str, k: int) -> list[str]:
    q = set(query.lower().split())
    scored = sorted(_DOCS, key=lambda d: -len(q & set(d[1].lower().split())))
    return [d[1] for d in scored[:k]]


def retrieve(query: str, k: int = 2) -> list[str]:
    """Return the k most relevant documentation snippets for a query."""
    col = _get_collection()
    if col is not None:
        try:
            res = col.query(query_texts=[query], n_results=min(k, len(_DOCS)))
            docs = res.get("documents") if res else None
            if docs and docs[0]:
                return docs[0]
        except Exception:
            pass
    return _keyword_fallback(query, k)


In [ ]:
import time

AGENTS = [
    {"id": "planner",   "name": "Planner",   "emoji": "\U0001F9E0", "accent": "#8b5cf6"},
    {"id": "coder",     "name": "Coder",     "emoji": "\U0001F4BB", "accent": "#25d7f0"},
    {"id": "executor",  "name": "Executor",  "emoji": "\u2699\ufe0f", "accent": "#38bdf8"},
    {"id": "critic",    "name": "Critic",    "emoji": "\U0001F527", "accent": "#fbbf24"},
    {"id": "automl",    "name": "AutoML",    "emoji": "\u2728",       "accent": "#9ae64a"},
    {"id": "explainer", "name": "Explainer", "emoji": "\U0001F50D", "accent": "#e879f9"},
    {"id": "reporter",  "name": "Reporter",  "emoji": "\U0001F4DD", "accent": "#fb7185"},
]
AGENT_IDS = [a["id"] for a in AGENTS]

def run_real_sync(df, target, goal, task, emit, llm=None):
    if llm is None:
        llm = get_llm("planner")
    rows, cols = df.shape
    ds_info = {"name": "uploaded dataset", "rows": int(rows), "cols": int(cols), "task": task, "target": target, "columns": list(df.columns)[:40]}
    goal = goal or ("Analyze " + target + " and explain the key drivers.")

    emit("planner", status="active")
    emit("planner", log={"text": "> objective: " + goal, "kind": "muted"}); time.sleep(0.3)
    emit("planner", log={"text": "dataset: " + format(rows, ",") + " rows x " + str(cols) + " cols | target: " + target, "kind": "info"})
    curated = ["1. load & validate dataset", "2. profile & clean columns", "3. encode features", "4. train model to predict " + target, "5. evaluate on held-out data", "6. explain drivers with SHAP"]
    plan = [l.strip() for l in llm.complete("planner", {"goal": goal, "dataset": ds_info, "plan": curated}).splitlines() if l.strip()][:6] or curated
    for l in plan:
        time.sleep(0.16); emit("planner", log={"text": l, "kind": "code"})
    emit("planner", status="done")

    emit("coder", status="active")
    emit("coder", log={"text": "retrieving library docs (ChromaDB RAG)...", "kind": "muted"})
    docs = retrieve(goal + " " + target, 2)
    for d in docs[:2]:
        emit("coder", log={"text": "  doc: " + d[:66] + "...", "kind": "muted"})
    curated = "print('rows:', df.shape[0], 'cols:', df.shape[1])\nprint('column means:')\nprint(df.mean(numeric_only=False).round(2).to_string())"
    curated_fix = curated.replace("numeric_only=False", "numeric_only=True")
    gen = strip_code_fences(llm.complete("coder", {"dataset": ds_info, "step": "profile the dataset", "code": [curated], "docs": docs})) or curated
    for l in gen.splitlines()[:8]:
        time.sleep(0.15); emit("coder", log={"text": l, "kind": "code"})
    emit("coder", status="done")

    emit("executor", status="active")
    emit("executor", log={"text": "> RestrictedPython sandbox  (fs: off | net: off)", "kind": "muted"}); time.sleep(0.3)
    emit("executor", log={"text": "loaded " + format(rows, ",") + " rows x " + str(cols) + " cols", "kind": "info"})
    current = gen; ran_ok = False
    for attempt in range(5):
        res = run_in_sandbox(current, {"df": df.copy()})
        if res.ok:
            for l in (res.stdout or "").strip().splitlines()[:8]:
                emit("executor", log={"text": "  " + l, "kind": "code"})
            emit("executor", log={"text": "OK generated code executed", "kind": "ok"}); ran_ok = True; break
        emit("executor", log={"text": "Traceback: " + res.error, "kind": "err"}); emit("executor", status="error")
        emit("critic", status="active"); emit("critic", log={"text": "reading traceback...", "kind": "muted"}); time.sleep(0.4)
        fixed = strip_code_fences(llm.complete("critic", {"error": res.error, "code": current, "fix": curated_fix}).strip()) or curated_fix
        emit("critic", log={"text": "patched the code, retry " + str(attempt + 1) + "/5", "kind": "warn"}); emit("critic", status="done")
        current = fixed
        emit("executor", status="active"); emit("executor", log={"text": "> re-running patched code...", "kind": "muted"})
    if not ran_ok:
        emit("executor", log={"text": "could not auto-fix; using the trusted engine", "kind": "warn"})
    try:
        _, fixes = clean(df, target)
    except Exception:
        fixes = []
    for fx in fixes[:4]:
        emit("executor", log={"text": "auto-clean: " + fx, "kind": "muted"})
    emit("executor", log={"text": "training models (FLAML AutoML, ~20s)...", "kind": "muted"})
    results = analyze_dataframe(df, target, task, 20)
    emit("executor", log={"text": "OK trained on " + format(results["_rows"], ",") + " rows", "kind": "ok"}); emit("executor", status="done")

    emit("automl", status="active"); hv = results["headline"]
    emit("automl", log={"text": "  best: " + results["bestModel"] + "  " + hv["label"] + " " + hv["value"] + "  * best", "kind": "ok"})
    emit("automl", log={"text": "task detected -> " + results["taskLabel"], "kind": "info"}); emit("automl", status="done")

    emit("explainer", status="active"); emit("explainer", log={"text": "computing " + results["barsTitle"] + "...", "kind": "muted"}); time.sleep(0.3)
    for b in results["bars"][:4]:
        sign = "-" if b.get("sign") == -1 else "+"; time.sleep(0.2)
        emit("explainer", log={"text": "  " + str(b["label"]).ljust(18) + " " + sign + str(round(b["value"], 2)), "kind": "code"})
    emit("explainer", status="done")

    emit("reporter", status="active"); emit("reporter", log={"text": "drafting business narrative...", "kind": "muted"}); time.sleep(0.4)
    drivers = ", ".join(str(d["feature"]) for d in results["_drivers"]); metrics = ", ".join(m["label"] + " " + m["value"] for m in results["metrics"])
    rtext = llm.complete("reporter", {"report": results["report"], "recommendation": results["recommendation"], "dataset": ds_info, "drivers": drivers, "metrics": metrics})
    if not llm.is_mock and rtext:
        paras = [p.strip() for p in rtext.split("\n\n") if p.strip()]
        if paras:
            results["report"] = paras[:-1] or paras; results["recommendation"] = paras[-1].replace("Recommendation:", "").strip()
    emit("reporter", log={"text": "OK report generated", "kind": "ok"}); emit("reporter", status="done")
    return results

print("real-run ready:", " -> ".join(AGENT_IDS))

In [ ]:
import gradio as gr, threading, queue, html as _html, pandas as pd

LOG_COLOR = {"info": "#c9d6ef", "code": "#9fb0cc", "ok": "#9ae64a", "err": "#fb7185", "warn": "#fbbf24", "muted": "#76859f"}
ACCENT = {a["id"]: a["accent"] for a in AGENTS}

def status_html(statuses, fixed=False):
    out = "<div style=\'display:flex;gap:6px;flex-wrap:wrap\'>"
    for a in AGENTS:
        st = statuses.get(a["id"], "queued")
        icon = {"queued": "o", "active": "...", "done": "OK", "error": "X"}[st]
        col = {"queued": "#76859f", "active": a["accent"], "done": "#9ae64a", "error": "#fb7185"}[st]
        bg = "#0e1426" if st == "queued" else a["accent"] + "22"
        tag = " <span style=\'color:#fbbf24;font-size:9px\'>fixed</span>" if (a["id"] == "critic" and fixed) else ""
        out += "<div style=\'border:1px solid " + col + "55;background:" + bg + ";border-radius:10px;padding:6px 10px;font-size:12px;color:#c9d6ef\'>" + a["emoji"] + " " + a["name"] + " <b style=\'color:" + col + "\'>" + icon + "</b>" + tag + "</div>"
    return out + "</div>"

def log_html(lines):
    rows = ""
    for stage, text, kind in lines:
        rows += "<div style=\'display:flex;gap:8px\'><span style=\'color:" + ACCENT.get(stage, "#76859f") + "\'>|</span><span style=\'color:" + LOG_COLOR.get(kind, "#c9d6ef") + ";white-space:pre-wrap\'>" + _html.escape(text) + "</span></div>"
    if not rows:
        rows = "<span style=\'color:#76859f\'>waiting...</span>"
    return "<div style=\'font-family:monospace;font-size:12.5px;background:#06080f;border:1px solid #1a2440;border-radius:10px;padding:12px;height:330px;overflow:auto\'>" + rows + "</div>"

def _bars(bars):
    out = ""
    for b in bars:
        col = "#25d7f0" if b.get("sign", 1) >= 0 else "#fb7185"
        w = str(int(b["value"] * 100))
        out += "<div style=\'display:flex;align-items:center;gap:8px;margin:5px 0\'><span style=\'width:150px;text-align:right;font-size:12px;color:#c9d6ef\'>" + str(b["label"]) + "</span><div style=\'flex:1;background:#0e1426;border-radius:6px;height:10px\'><div style=\'width:" + w + "%;height:10px;background:" + col + ";border-radius:6px\'></div></div></div>"
    return out

def _dist(items):
    out = ""
    for d in items:
        w = str(int(d["value"] * 100))
        out += "<div style=\'margin:6px 0\'><div style=\'display:flex;justify-content:space-between;font-size:12px;color:#c9d6ef\'><span>" + str(d["label"]) + "</span><span style=\'color:#76859f\'>" + str(d["display"]) + "</span></div><div style=\'background:#0e1426;border-radius:6px;height:9px;margin-top:3px\'><div style=\'width:" + w + "%;height:9px;background:#8b5cf6;border-radius:6px\'></div></div></div>"
    return out

def results_html(r, accent="#25d7f0"):
    if not r:
        return "<div style=\'color:#76859f;padding:18px\'>No results yet - upload a CSV and run.</div>"
    chips = ""
    for m in r["metrics"]:
        chips += "<div style=\'border:1px solid #1a2440;background:#0a1020;border-radius:10px;padding:10px 14px;text-align:center\'><div style=\'font-size:20px;font-weight:700;color:#fff\'>" + m["value"] + "</div><div style=\'font-size:10px;color:#76859f;text-transform:uppercase\'>" + m["label"] + "</div></div>"
    head = "<div style=\'display:flex;gap:10px;align-items:center;margin-bottom:12px\'><span style=\'font-size:30px;font-weight:700;color:" + accent + "\'>" + r["headline"]["value"] + "</span><span style=\'font-size:11px;color:#76859f\'>" + r["headline"]["label"] + "</span><span style=\'margin-left:auto;font-size:12px;color:#c9d6ef\'>best: <b>" + r["bestModel"] + "</b></span></div>"
    paras = ""
    for p in r["report"]:
        paras += "<p style=\'color:#c9d6ef;font-size:13px;line-height:1.6;margin:6px 0\'>" + _html.escape(p) + "</p>"
    rec = "<div style=\'border:1px solid " + accent + "55;background:" + accent + "11;border-radius:10px;padding:12px;margin-top:8px\'><b style=\'color:" + accent + ";font-size:11px;text-transform:uppercase\'>Recommendation</b><div style=\'color:#fff;font-size:13px;margin-top:4px\'>" + _html.escape(r["recommendation"]) + "</div></div>"
    return ("<div style=\'background:#0a1020;border:1px solid #1a2440;border-radius:14px;padding:16px\'>" + head +
            "<div style=\'display:flex;gap:8px;flex-wrap:wrap;margin-bottom:14px\'>" + chips + "</div>" +
            "<div style=\'color:#76859f;font-size:11px;text-transform:uppercase;margin:8px 0\'>" + r["barsTitle"] + "</div>" + _bars(r["bars"]) +
            "<div style=\'color:#76859f;font-size:11px;text-transform:uppercase;margin:14px 0 4px\'>" + r["distTitle"] + "</div>" + _dist(r["dist"]) +
            "<div style=\'color:#76859f;font-size:11px;text-transform:uppercase;margin:14px 0 4px\'>Business report</div>" + paras + rec + "</div>")

def on_upload(path):
    if not path:
        return gr.update(choices=[], value=None)
    try:
        cols = list(pd.read_csv(path, nrows=5).columns)
        return gr.update(choices=cols, value=cols[-1] if cols else None)
    except Exception:
        return gr.update(choices=[], value=None)

def run_and_stream(path, goal, target, task, provider, model, key):
    if not path:
        yield "<div style=\'color:#fb7185\'>Upload a CSV first.</div>", "", ""
        return
    if not target:
        yield "<div style=\'color:#fb7185\'>Pick a target column.</div>", "", ""
        return
    df = pd.read_csv(path)
    llm = make_llm(provider, model, key) if key else get_llm("planner")
    q = queue.Queue()
    def emit(stage, status=None, log=None):
        q.put(("event", stage, status, log))
    def worker():
        try:
            res = run_real_sync(df, target, goal, task, emit, llm); q.put(("result", res))
        except Exception as e:
            q.put(("error", str(e)))
        finally:
            q.put(None)
    threading.Thread(target=worker, daemon=True).start()
    statuses = {a: "queued" for a in AGENT_IDS}; lines = []; results = None; fixed = False
    yield status_html(statuses), log_html(lines), results_html(None)
    while True:
        item = q.get()
        if item is None:
            break
        if item[0] == "event":
            _, stage, status, log = item
            if status:
                statuses[stage] = status
                if stage == "critic" and status == "done": fixed = True
            if log:
                lines.append((stage, log["text"], log.get("kind", "info")))
            yield status_html(statuses, fixed), log_html(lines), results_html(results)
        elif item[0] == "result":
            results = item[1]
        elif item[0] == "error":
            lines.append(("reporter", "(error: " + item[1] + ")", "err"))
            yield status_html(statuses, fixed), log_html(lines), results_html(results)
    yield status_html(statuses, fixed), log_html(lines), results_html(results)

HEADER = "<div style=\'text-align:center;padding:6px\'><h1 style=\'margin:0;color:#fff\'>Helix - Real Analysis on your data</h1><p style=\'color:#76859f;margin:4px\'>Upload a CSV, pick the target, and watch 7 agents run a genuine FLAML + SHAP analysis with self-correction.</p></div>"

with gr.Blocks(title="Helix - Real Data", css=".gradio-container{max-width:1080px !important}", theme=gr.themes.Base()) as demo:
    gr.HTML(HEADER)
    with gr.Row():
        with gr.Column(scale=1):
            file_in = gr.File(label="1 - Upload CSV", file_types=[".csv"], type="filepath")
            target_in = gr.Dropdown(choices=[], label="2 - Target column (what to predict)")
            task_in = gr.Radio(["auto", "classification", "regression"], value="auto", label="Task")
            goal_in = gr.Textbox(value="Analyze the target and explain the key drivers.", lines=2, label="Goal")
            provider_in = gr.Dropdown(["groq", "openai", "deepseek", "anthropic", "mistral", "openrouter", "gemini"], value="groq", label="AI provider")
            model_in = gr.Textbox(value="", placeholder="(optional) model id e.g. llama-3.3-70b-versatile", label="Model")
            key_in = gr.Textbox(value="", type="password", placeholder="(optional) paste key for real LLM", label="API key")
            run_btn = gr.Button("Run real analysis", variant="primary")
        with gr.Column(scale=2):
            status_out = gr.HTML()
            log_out = gr.HTML()
    results_out = gr.HTML()

    file_in.change(on_upload, file_in, target_in)
    run_btn.click(run_and_stream, [file_in, goal_in, target_in, task_in, provider_in, model_in, key_in], [status_out, log_out, results_out])

print("Gradio app built - run the next cell to launch.")

In [ ]:
demo.queue().launch(share=True, debug=False)